In [ ]:
import sys

from hydra.utils import instantiate
from einops import rearrange
from omegaconf import OmegaConf
import torch
import os
import sys

sys.path.append(os.path.abspath("/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models")
                )
# Register custom resolver to handle multiplication in OmegaConf interpolation
OmegaConf.register_new_resolver("mul", lambda x, y: float(x) * float(y))
OmegaConf.register_new_resolver("div", lambda a, b: float(a) / float(b))

In [ ]:
complete_cfg = OmegaConf.load("/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/obs_ad_de_aug_latent_mt/2025-09-02/04-47-50/0/.hydra/config.yaml")
complete_val_dataloader = instantiate(complete_cfg.val_data_loader)

In [ ]:
snapshot_dir = "/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/obs_ad_de_aug_latent_mt/2025-09-02/04-47-50/0"
config_path = f"{snapshot_dir}/.hydra"
print(config_path)

cfg = OmegaConf.load(config_path + "/config.yaml")

In [ ]:

model = instantiate(cfg.model)
model = model.cuda().eval()
snapshot = torch.load(f"{snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
model.load_state_dict(snapshot["model"])
val_dataloader = instantiate(cfg.val_data_loader)

In [27]:
# bs_snapshot_dir = "/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/gt_droid_egodex/2025-07-28/23-50-47/0"
bs_snapshot_dir = "/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/gt_de/2025-09-01/03-18-27/0"
bs_config_path = f"{bs_snapshot_dir}/.hydra"
print(bs_config_path)

bs_cfg = OmegaConf.load(bs_config_path + "/config.yaml")

/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/gt_de/2025-09-01/03-18-27/0/.hydra


In [28]:
bs_model = instantiate(bs_cfg.model)
bs_model = bs_model.cuda().eval()
snapshot = torch.load(f"{bs_snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
bs_model.load_state_dict(snapshot["model"])

Checkpoint downloaded to /home/ravenhuang/.cache/huggingface/hub/models--facebook--robotics-world-models/snapshots/632c46831fcb91de52283ab22550eb94070874ad/mido-jepa-wm-rope/vitg16-vitlpred-proprio-noaug-leftcam-8fpc-4hz-2ar-latest.pth.tar
loaded pretrained target decoder with msg: <All keys matched successfully>
VisionTransformer(
  (patch_embed): PatchEmbed3D(
    (proj): Conv3d(3, 1408, kernel_size=(2, 16, 16), stride=(2, 16, 16))
  )
  (blocks): ModuleList(
    (0-39): 40 x Block(
      (norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
      (attn): RoPEAttention(
        (qkv): Linear(in_features=1408, out_features=4224, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1408, out_features=1408, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=1408, ou

<All keys matched successfully>

In [ ]:
bs_val_dataloader = instantiate(bs_cfg.val_data_loader)
simpler_dataloader = bs_val_dataloader[0]

In [ ]:
bs_val_dataloader[0].dataset.datasets.keys()

In [ ]:
simpler_zs_bs = []
simpler_zs = []

for i, simpler_batch in enumerate(simpler_dataloader):
    rgb = simpler_batch["rgb"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z_bs = bs_model._forward_action_encode( simpler_batch["actions"].cuda(), 
                                    simpler_batch["morphology_index"].cuda(),
                                    simpler_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    simpler_zs_bs.append(z_bs)
    
    z = model._forward_action_encode( simpler_batch["actions"].cuda(), 
                                    simpler_batch["morphology_index"].cuda(),
                                    simpler_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    simpler_zs.append(z)
    if i > 100:
        break

In [29]:
droid_dataloader = complete_val_dataloader[0]
egodex_dataloader = complete_val_dataloader[1]
mpk_dataloader = complete_val_dataloader[3]

In [36]:
droid_zs = []
droid_zs_bs = []
for i, droid_batch in enumerate(droid_dataloader):
    rgb = droid_batch["rgb"].cuda()
    actions = droid_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions).detach().cpu().numpy()
    droid_zs.append(z)
    
    z = bs_model._forward_action_encode( droid_batch["actions"].cuda(), 
                                    droid_batch["morphology_index"].cuda(),
                                    droid_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    droid_zs_bs.append(z)
    if i > 100:
        break

egodex_zs = []
egodex_zs_bs = []
for i, egodex_batch in enumerate(egodex_dataloader):
    rgb = egodex_batch["rgb"].cuda()
    actions = egodex_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions).detach().cpu().numpy()
    egodex_zs.append(z)
    
    z = bs_model._forward_action_encode( egodex_batch["actions"].cuda(), 
                                    egodex_batch["morphology_index"].cuda(),
                                    egodex_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    egodex_zs_bs.append(z)
    if i > 100:
        break

/home/ravenhuang/miniconda3/envs/robot_world_models/lib/python3.10/contextlib.py:103: FutureWarning:

`torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.



In [ ]:
droid_zs = []
for i, droid_batch in enumerate(droid_dataloader):
    rgb = droid_batch["rgb"].cuda()
    actions = droid_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions).detach().cpu().numpy()
    droid_zs.append(z)
    if i > 100:
        break

egodex_zs = []
for i, egodex_batch in enumerate(egodex_dataloader):
    rgb = egodex_batch["rgb"].cuda()
    actions = egodex_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions).detach().cpu().numpy()
    egodex_zs.append(z)
    if i > 100:
        break
mpk_zs = []
for i, mpk_batch in enumerate(mpk_dataloader):
    rgb = mpk_batch["rgb"].cuda()
    actions = mpk_batch["actions"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x, actions).detach().cpu().numpy()
    mpk_zs.append(z)
    if i > 100:
        break

In [ ]:
murp_zs = []
for i, murp_batch in enumerate(murp_dataloader):
    rgb = murp_batch["rgb"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = bs_model._forward_action_encode( murp_batch["actions"].cuda(), 
                                    murp_batch["morphology_index"].cuda(),
                                    murp_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    murp_zs.append(z)
    if i > 100:
        break

In [ ]:
droid_zs_bs = []
for i, droid_batch in enumerate(droid_dataloader):
    z = bs_model._forward_action_encode( droid_batch["actions"].cuda(), 
                                    droid_batch["morphology_index"].cuda(),
                                    droid_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    droid_zs_bs.append(z)
    if i > 50:
        break
egodex_zs_bs = []
for i, egodex_batch in enumerate(egodex_dataloader):
    z = bs_model._forward_action_encode( egodex_batch["actions"].cuda(), 
                                    egodex_batch["morphology_index"].cuda(),
                                    egodex_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    egodex_zs_bs.append(z)
    if i > 50:
        break

mpk_zs_bs = []
for i, mpk_batch in enumerate(mpk_dataloader):
    z = bs_model._forward_action_encode( mpk_batch["actions"].cuda(), 
                                    mpk_batch["morphology_index"].cuda(),
                                    mpk_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    mpk_zs_bs.append(z)
    if i > 50:
        break

In [ ]:
mpk_batch['rgb'].shape

In [ ]:
import mediapy as media
media.write_video("droid_0.mp4", rearrange(droid_batch['rgb'][0].cpu().numpy(), "t c h w -> t h w c"), fps=10)

In [ ]:
hot3d_zs = []
for i, hot3d_batch in enumerate(hot3d_dataloader):
    rgb = hot3d_batch["rgb"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x).detach().cpu().numpy()
    hot3d_zs.append(z)
    if i > 50:
        break



In [ ]:
hot3d_zs_bs = []
for i, hot3d_batch in enumerate(hot3d_dataloader):
    z = bs_model._forward_action_encode( hot3d_batch["actions"].cuda(), 
                                    hot3d_batch["morphology_index"].cuda(),
                                    hot3d_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    hot3d_zs_bs.append(z)
    if i > 50:
        break

In [ ]:
import numpy as np

In [ ]:
simpler_zs_c = np.concatenate(simpler_zs, axis=0)
simpler_zs_bs_c = np.concatenate(simpler_zs_bs, axis=0)
flatten_simpler_zs = rearrange(simpler_zs_c, "b t a m -> (b t) (a m)")
flatten_simpler_zs_bs = rearrange(simpler_zs_bs_c, "b t a m -> (b t) (a m)")

In [ ]:
droid_zs_c = np.concatenate(droid_zs, axis=0)
egodex_zs_c = np.concatenate(egodex_zs, axis=0)
mpk_zs_c = np.concatenate(mpk_zs, axis=0)

In [ ]:
droid_zs_bs_c = np.concatenate(droid_zs_bs, axis=0)
egodex_zs_bs_c = np.concatenate(egodex_zs_bs, axis=0)
mpk_zs_bs_c = np.concatenate(mpk_zs_bs, axis=0)

In [ ]:
flatten_droid_zs = rearrange(droid_zs_c, "b t a m -> (b t) (a m)")
flatten_egodex_zs = rearrange(egodex_zs_c, "b t a m -> (b t) (a m)")
flatten_mpk_zs = rearrange(mpk_zs_c, "b t a m -> (b t) (a m)")

In [ ]:
flatten_droid_zs_bs = rearrange(droid_zs_bs_c, "b t a m -> (b t a) m")
flatten_egodex_zs_bs = rearrange(egodex_zs_bs_c, "b t a m -> (b t a) m")
flatten_mpk_zs_bs = rearrange(mpk_zs_bs_c, "b t a m -> (b t a) m")

In [ ]:
embeddings = np.stack([flatten_egodex_zs, flatten_droid_zs, flatten_mpk_zs], axis=0)
# embeddings = np.stack([flatten_droid_zs, flatten_egodex_zs], axis=0)


In [ ]:
bs_embeddings = np.stack([flatten_droid_zs_bs, flatten_egodex_zs_bs, flatten_mpk_zs_bs], axis=0)
# bs_embeddings = np.stack([flatten_droid_zs_bs, flatten_egodex_zs_bs], axis=0)


In [ ]:
import umap
import numpy as np
import plotly.express as px
import torch

def plot_umap_3d_interactive(embeddings, color_labels=None):
    """
    embeddings: torch.Tensor or np.ndarray of shape (S, B, M)
    color_labels: optional array of shape (S*B,) for coloring
    """
    # Convert to numpy
    if torch.is_tensor(embeddings):
        embeddings = embeddings.detach().cpu().numpy()

    S, B, M = embeddings.shape
    embeddings_flat = embeddings.reshape(S * B, M)

    # Default color labels = data source index
    if color_labels is None:
        color_labels = np.repeat(np.arange(S), B)

    # Run UMAP
    reducer = umap.UMAP(n_components=3, random_state=42)
    embeddings_3d = reducer.fit_transform(embeddings_flat)

    # Create interactive plot
    fig = px.scatter_3d(
        x=embeddings_3d[:, 0],
        y=embeddings_3d[:, 1],
        z=embeddings_3d[:, 2],
        color=color_labels.astype(str),  # plotly requires string or category
        labels={'color': 'Data Source'},
        title="Interactive 3D UMAP"
    )
    fig.update_traces(marker=dict(size=3, opacity=0.7))
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
    fig.show()


In [ ]:
plot_umap_3d_interactive(flatten_simpler_zs[None])

In [ ]:
plot_umap_3d_interactive(flatten_simpler_zs_bs[None])

In [ ]:
S, B, M = embeddings.shape

In [ ]:
droid_me = model.morphology_tokens.weight[0]

In [ ]:
egodex_me = model.morphology_tokens.weight[2]

In [ ]:
plot_umap_3d_interactive(embeddings, color_labels=np.repeat(np.arange(3), embeddings.shape[1]))

In [ ]:
plot_umap_3d_interactive(embeddings[...,:M//2], color_labels=np.repeat(np.arange(3), embeddings.shape[1]))

In [ ]:
embed_pm = torch.stack([egodex_me, droid_me,droid_me ], dim=0).detach().cpu().numpy()

In [ ]:
new_embed = embed_pm[:,None] + embeddings

In [ ]:
plot_umap_3d_interactive(new_embed[...,:M//2], color_labels=np.repeat(np.arange(3), embeddings.shape[1]))

In [ ]:
plot_umap_3d_interactive(bs_embeddings, color_labels=np.repeat(np.arange(3), bs_embeddings.shape[1]))


In [ ]:
embeddings.shape

In [ ]:
flatten_droid_zs.shape

In [ ]:
droid_internal = rearrange(droid_zs_c, "b t a m -> (b t) a m")

In [ ]:
droid_internal.shape

In [ ]:
plot_umap_3d_interactive(droid_internal[:10], color_labels=np.repeat(np.arange(10), droid_internal[:10].shape[1]))
